## 04 Campaign Generation

Use enriched product attributes and customer segment strategies to generate segment-specific campaign recommendations.

Output:

- campaign_recommendations.csv


#### Load Data


In [1]:
import os
import pandas as pd
import numpy as np

In [2]:
PROCESSED_DIR = "../data/processed/hm"

articles = pd.read_csv(os.path.join(PROCESSED_DIR, "articles_enriched.csv"))
customer_segments = pd.read_csv(os.path.join(PROCESSED_DIR, "customer_segments.csv"))
segment_strategy = pd.read_csv(os.path.join(PROCESSED_DIR, "segment_strategy.csv"))

print(articles.shape)
print(customer_segments.shape)
print(segment_strategy.shape)

(105542, 37)
(317897, 27)
(5, 4)


In [3]:
articles.head()

,article_id,product_code,product_name,product_type_no,product_type,product_group,graphical_appearance_no,graphical_appearance,colour_group_code,color_group,...,product_purchase_count,unique_customer_count,avg_selling_price,style,occasion,material_hint,target_audience,selling_points,marketing_keywords,copy_angle
0,108775015,108775,Strap top,253,Vest top,Garment Upper body,1010016,Solid,9,Black,...,175.0,172.0,0.008139,everyday,daily_wear,unknown,Ladieswear,"['Easy to style Vest top', 'Versatile for dail...","['everyday', 'daily_wear', 'Black']",everyday and daily_wear focused
1,108775044,108775,Strap top,253,Vest top,Garment Upper body,1010016,Solid,10,White,...,116.0,116.0,0.008196,everyday,daily_wear,unknown,Ladieswear,"['Easy to style Vest top', 'Versatile for dail...","['everyday', 'daily_wear', 'White']",everyday and daily_wear focused
2,108775051,108775,Strap top (1),253,Vest top,Garment Upper body,1010017,Stripe,11,Off White,...,2.0,2.0,0.004559,everyday,daily_wear,unknown,Ladieswear,"['Easy to style Vest top', 'Versatile for dail...","['everyday', 'daily_wear', 'Off White']",everyday and daily_wear focused
3,110065001,110065,OP T-shirt (Idro),306,Bra,Underwear,1010016,Solid,9,Black,...,19.0,19.0,0.020756,elegant,work_or_outing,unknown,Ladieswear,"['Easy to style Bra', 'Versatile for work_or_o...","['elegant', 'work_or_outing', 'Black']",elegant and work_or_outing focused
4,110065002,110065,OP T-shirt (Idro),306,Bra,Underwear,1010016,Solid,10,White,...,11.0,11.0,0.016932,elegant,work_or_outing,unknown,Ladieswear,"['Easy to style Bra', 'Versatile for work_or_o...","['elegant', 'work_or_outing', 'White']",elegant and work_or_outing focused


In [4]:
segment_strategy

,customer_segment,segment_description,recommended_strategy,copy_angle
0,High-Value One-Time Buyers,Customers who purchased rarely but spent relat...,Reactivation campaigns with premium positioning,"quality, exclusivity, worth coming back for"
1,Inactive Budget Shoppers,Low-spending customers with low engagement and...,"Win-back discounts, simple value offers, clear...","affordable, practical, low-commitment"
2,Engaged Budget Shoppers,Price-sensitive customers who are still highly...,"Deal-focused campaigns, bundle offers, persona...","value-driven, timely, deal-oriented"
3,Regular Shoppers,Moderately active customers with steady purcha...,"Cross-sell recommendations, seasonal edits, ev...","easy, relevant, everyday style"
4,Loyal High-Value Customers,"Frequent, high-spending customers with more re...","VIP rewards, early access, premium recommendat...","exclusive, personalized, rewarding"


#### Prepare Product Candidates


In [5]:
# Keep products with enrichment fields available
required_cols = [
    "article_id",
    "product_name",
    "product_type",
    "product_group",
    "color_group",
    "index_group",
    "garment_group",
    "description",
    "style",
    "occasion",
    "material_hint",
    "target_audience",
    "selling_points",
    "marketing_keywords",
    "copy_angle",
    "product_purchase_count",
    "unique_customer_count",
    "avg_selling_price"
]

products = articles[required_cols].copy()

products.head()

,article_id,product_name,product_type,product_group,color_group,index_group,garment_group,description,style,occasion,material_hint,target_audience,selling_points,marketing_keywords,copy_angle,product_purchase_count,unique_customer_count,avg_selling_price
0,108775015,Strap top,Vest top,Garment Upper body,Black,Ladieswear,Jersey Basic,Jersey top with narrow shoulder straps.,everyday,daily_wear,unknown,Ladieswear,"['Easy to style Vest top', 'Versatile for dail...","['everyday', 'daily_wear', 'Black']",everyday and daily_wear focused,175.0,172.0,0.008139
1,108775044,Strap top,Vest top,Garment Upper body,White,Ladieswear,Jersey Basic,Jersey top with narrow shoulder straps.,everyday,daily_wear,unknown,Ladieswear,"['Easy to style Vest top', 'Versatile for dail...","['everyday', 'daily_wear', 'White']",everyday and daily_wear focused,116.0,116.0,0.008196
2,108775051,Strap top (1),Vest top,Garment Upper body,Off White,Ladieswear,Jersey Basic,Jersey top with narrow shoulder straps.,everyday,daily_wear,unknown,Ladieswear,"['Easy to style Vest top', 'Versatile for dail...","['everyday', 'daily_wear', 'Off White']",everyday and daily_wear focused,2.0,2.0,0.004559
3,110065001,OP T-shirt (Idro),Bra,Underwear,Black,Ladieswear,"Under-, Nightwear","Microfibre T-shirt bra with underwired, moulde...",elegant,work_or_outing,unknown,Ladieswear,"['Easy to style Bra', 'Versatile for work_or_o...","['elegant', 'work_or_outing', 'Black']",elegant and work_or_outing focused,19.0,19.0,0.020756
4,110065002,OP T-shirt (Idro),Bra,Underwear,White,Ladieswear,"Under-, Nightwear","Microfibre T-shirt bra with underwired, moulde...",elegant,work_or_outing,unknown,Ladieswear,"['Easy to style Bra', 'Versatile for work_or_o...","['elegant', 'work_or_outing', 'White']",elegant and work_or_outing focused,11.0,11.0,0.016932


In [7]:
# Fill missing numeric values
products["product_purchase_count"] = products["product_purchase_count"].fillna(0)
products["unique_customer_count"] = products["unique_customer_count"].fillna(0)
products["avg_selling_price"] = products["avg_selling_price"].fillna(0)

#### Add Simulated Inventory


In [8]:
np.random.seed(42)

# Simulate inventory levels
products["inventory_level"] = np.random.choice(
    ["low", "medium", "high"],
    size=len(products),
    p=[0.25, 0.45, 0.30]
)

# Convert into numeric score
products["inventory_score"] = products["inventory_level"].map({
    "low": 0.3,
    "medium": 0.7,
    "high": 1.0
})

products[["article_id", "product_name", "inventory_level", "inventory_score"]].head()

,article_id,product_name,inventory_level,inventory_score
0,108775015,Strap top,medium,0.7
1,108775044,Strap top,high,1.0
2,108775051,Strap top (1),high,1.0
3,110065001,OP T-shirt (Idro),medium,0.7
4,110065002,OP T-shirt (Idro),low,0.3


#### Create Product Scores


In [ ]:
# Normalizes each score between 0 and 1
def min_max_scale(series):
    if series.max() == series.min():
        return pd.Series(0.5, index=series.index)
    return (series - series.min()) / (series.max() - series.min())

products["popularity_score"] = min_max_scale(products["product_purchase_count"])
products["customer_reach_score"] = min_max_scale(products["unique_customer_count"])
products["price_score"] = min_max_scale(products["avg_selling_price"])

In [10]:
# Build product score
products["base_product_score"] = (
    0.50 * products["popularity_score"]
    + 0.30 * products["customer_reach_score"]
    + 0.20 * products["inventory_score"]
)

products[
    [
        "article_id",
        "product_name",
        "popularity_score",
        "customer_reach_score",
        "inventory_score",
        "base_product_score"
    ]
].head()

,article_id,product_name,popularity_score,customer_reach_score,inventory_score,base_product_score
0,108775015,Strap top,0.209581,0.207229,0.7,0.306959
1,108775044,Strap top,0.138922,0.139759,1.0,0.311389
2,108775051,Strap top (1),0.002395,0.002410,1.0,0.201920
3,110065001,OP T-shirt (Idro),0.022754,0.022892,0.7,0.158245
4,110065002,OP T-shirt (Idro),0.013174,0.013253,0.3,0.070563


#### Segment-Product Matching

This tells the system what each customer segment tends to prefer.


In [11]:
# Rule-based “marketing strategy layer"
segment_profiles = {
    "High-Value One-Time Buyers": {
        "preferred_price": "high",
        "preferred_styles": ["elegant", "premium", "everyday"],
        "campaign_goal": "reactivation"
    },
    "Inactive Budget Shoppers": {
        "preferred_price": "low",
        "preferred_styles": ["casual", "everyday", "sporty"],
        "campaign_goal": "win_back"
    },
    "Engaged Budget Shoppers": {
        "preferred_price": "low",
        "preferred_styles": ["casual", "everyday", "streetwear", "sporty"],
        "campaign_goal": "conversion"
    },
    "Regular Shoppers": {
        "preferred_price": "medium",
        "preferred_styles": ["casual", "everyday", "elegant"],
        "campaign_goal": "repeat_purchase"
    },
    "Loyal High-Value Customers": {
        "preferred_price": "high",
        "preferred_styles": ["elegant", "premium", "trend", "streetwear"],
        "campaign_goal": "loyalty"
    }
}

In [12]:
# Turns normalized prices into categories
def assign_price_tier(price_score):
    if price_score < 0.33:
        return "low"
    elif price_score < 0.67:
        return "medium"
    else:
        return "high"

products["price_tier"] = products["price_score"].apply(assign_price_tier)

products[["article_id", "product_name", "avg_selling_price", "price_tier"]].head()

,article_id,product_name,avg_selling_price,price_tier
0,108775015,Strap top,0.008139,low
1,108775044,Strap top,0.008196,low
2,108775051,Strap top (1),0.004559,low
3,110065001,OP T-shirt (Idro),0.020756,low
4,110065002,OP T-shirt (Idro),0.016932,low


In [13]:

# Calculate segment match score
def calculate_segment_match(row, segment):
    profile = segment_profiles[segment]
    
    score = 0
    
    if row["price_tier"] == profile["preferred_price"]:
        score += 0.4
    
    if row["style"] in profile["preferred_styles"]:
        score += 0.4
    
    if row["inventory_level"] in ["medium", "high"]:
        score += 0.2
    
    return score

#### Generate Campaign Recommendations

For each customer segment:

- Copy all products
- Add the current segment name
- Calculate segment match score
- Calculate final campaign score
- Keep top 20 products


In [14]:
campaign_rows = []

for _, segment_row in segment_strategy.iterrows():
    segment = segment_row["customer_segment"]

    segment_products = products.copy()
    segment_products["customer_segment"] = segment
    segment_products["segment_match_score"] = segment_products.apply(
        lambda row: calculate_segment_match(row, segment),
        axis=1
    )

    # 60% product strength + 40% segment fit
    segment_products["campaign_score"] = (
        0.60 * segment_products["base_product_score"]
        + 0.40 * segment_products["segment_match_score"]
    )

    top_products = segment_products.sort_values(
        "campaign_score",
        ascending=False
    ).head(20)

    for _, row in top_products.iterrows():
        campaign_rows.append({
            "customer_segment": segment,
            "article_id": row["article_id"],
            "product_name": row["product_name"],
            "product_type": row["product_type"],
            "product_group": row["product_group"],
            "color_group": row["color_group"],
            "style": row["style"],
            "occasion": row["occasion"],
            "price_tier": row["price_tier"],
            "inventory_level": row["inventory_level"],
            "base_product_score": row["base_product_score"],
            "segment_match_score": row["segment_match_score"],
            "campaign_score": row["campaign_score"],
            "recommended_strategy": segment_row["recommended_strategy"],
            "copy_angle": segment_row["copy_angle"],
            "campaign_goal": segment_profiles[segment]["campaign_goal"]
        })

campaign_recommendations = pd.DataFrame(campaign_rows)

campaign_recommendations.head()

,customer_segment,article_id,product_name,product_type,product_group,color_group,style,occasion,price_tier,inventory_level,base_product_score,segment_match_score,campaign_score,recommended_strategy,copy_angle,campaign_goal
0,High-Value One-Time Buyers,706016001,Jade HW Skinny Denim TRS,Trousers,Garment Lower body,Black,streetwear,daily_wear,low,medium,0.940000,0.2,0.644000,Reactivation campaigns with premium positioning,"quality, exclusivity, worth coming back for",reactivation
1,High-Value One-Time Buyers,372860001,7p Basic Shaftless,Socks,Socks & Tights,Black,everyday,daily_wear,low,medium,0.653247,0.6,0.631948,Reactivation campaigns with premium positioning,"quality, exclusivity, worth coming back for",reactivation
2,High-Value One-Time Buyers,610776002,Tilly (1),T-shirt,Garment Upper body,Black,elegant,work_or_outing,low,high,0.652752,0.6,0.631651,Reactivation campaigns with premium positioning,"quality, exclusivity, worth coming back for",reactivation
3,High-Value One-Time Buyers,464297007,Greta Thong Mynta Low 3p,Underwear bottom,Underwear,Black,everyday,daily_wear,low,high,0.556727,0.6,0.574036,Reactivation campaigns with premium positioning,"quality, exclusivity, worth coming back for",reactivation
4,High-Value One-Time Buyers,156231001,Box 4p Tights,Underwear Tights,Socks & Tights,Black,everyday,daily_wear,low,high,0.533794,0.6,0.560276,Reactivation campaigns with premium positioning,"quality, exclusivity, worth coming back for",reactivation


#### Generate Campaign Messages


In [15]:
# Creates simple rule-based ad copy
def generate_campaign_message(row):
    segment = row["customer_segment"]
    product_name = row["product_name"]
    product_type = row["product_type"]
    style = row["style"]
    occasion = row["occasion"]
    copy_angle = row["copy_angle"]
    inventory = row["inventory_level"]

    if "Budget" in segment:
        return (
            f"Refresh your wardrobe with {product_name} — "
            f"a {style} {product_type} made for {occasion}, now with value-focused picks you'll love."
        )

    if "Loyal" in segment:
        return (
            f"An exclusive pick for your next favorite look: {product_name}, "
            f"a {style} {product_type} designed for {occasion}."
        )

    if "Inactive" in segment:
        return (
            f"Come back to something worth discovering — {product_name}, "
            f"a {style} {product_type} that makes everyday styling easier."
        )

    if "High-Value" in segment:
        return (
            f"Rediscover elevated style with {product_name}, "
            f"a {style} {product_type} selected for its {copy_angle} appeal."
        )

    return (
        f"Meet {product_name}: a {style} {product_type} for {occasion}, "
        f"recommended based on your shopping style."
    )

In [16]:
campaign_recommendations["campaign_message"] = campaign_recommendations.apply(
    generate_campaign_message,
    axis=1
)

campaign_recommendations[
    [
        "customer_segment",
        "product_name",
        "campaign_goal",
        "recommended_strategy",
        "campaign_message"
    ]
].head(10)

,customer_segment,product_name,campaign_goal,recommended_strategy,campaign_message
0,High-Value One-Time Buyers,Jade HW Skinny Denim TRS,reactivation,Reactivation campaigns with premium positioning,Rediscover elevated style with Jade HW Skinny ...
1,High-Value One-Time Buyers,7p Basic Shaftless,reactivation,Reactivation campaigns with premium positioning,Rediscover elevated style with 7p Basic Shaftl...
2,High-Value One-Time Buyers,Tilly (1),reactivation,Reactivation campaigns with premium positioning,"Rediscover elevated style with Tilly (1), a el..."
3,High-Value One-Time Buyers,Greta Thong Mynta Low 3p,reactivation,Reactivation campaigns with premium positioning,Rediscover elevated style with Greta Thong Myn...
4,High-Value One-Time Buyers,Box 4p Tights,reactivation,Reactivation campaigns with premium positioning,"Rediscover elevated style with Box 4p Tights, ..."
5,High-Value One-Time Buyers,Lazer Razer Brief,reactivation,Reactivation campaigns with premium positioning,Rediscover elevated style with Lazer Razer Bri...
6,High-Value One-Time Buyers,Luna skinny RW,reactivation,Reactivation campaigns with premium positioning,"Rediscover elevated style with Luna skinny RW,..."
7,High-Value One-Time Buyers,Luna skinny RW,reactivation,Reactivation campaigns with premium positioning,"Rediscover elevated style with Luna skinny RW,..."
8,High-Value One-Time Buyers,Scallop 5p Socks,reactivation,Reactivation campaigns with premium positioning,Rediscover elevated style with Scallop 5p Sock...
9,High-Value One-Time Buyers,Lea leather,reactivation,Reactivation campaigns with premium positioning,"Rediscover elevated style with Lea leather, a ..."


#### Add Promotion Channel


In [17]:
# Assigns a marketing channel based on segment
def recommend_channel(segment):
    if "Inactive" in segment:
        return "email"
    elif "Budget" in segment:
        return "push_notification"
    elif "Loyal" in segment:
        return "email_and_app_homepage"
    elif "High-Value" in segment:
        return "personalized_email"
    else:
        return "app_homepage"

In [18]:
campaign_recommendations["promotion_channel"] = campaign_recommendations[
    "customer_segment"
].apply(recommend_channel)

campaign_recommendations[["customer_segment", "promotion_channel"]].drop_duplicates()

,customer_segment,promotion_channel
0,High-Value One-Time Buyers,personalized_email
20,Inactive Budget Shoppers,email
40,Engaged Budget Shoppers,push_notification
60,Regular Shoppers,app_homepage
80,Loyal High-Value Customers,email_and_app_homepage


#### Create Segment-Level Campaign Summary


In [19]:
segment_campaign_summary = (
    campaign_recommendations
    .groupby("customer_segment")
    .agg(
        top_campaign_score=("campaign_score", "max"),
        avg_campaign_score=("campaign_score", "mean"),
        recommended_products=("article_id", "count"),
        top_strategy=("recommended_strategy", "first"),
        primary_channel=("promotion_channel", "first")
    )
    .reset_index()
)

segment_campaign_summary

,customer_segment,top_campaign_score,avg_campaign_score,recommended_products,top_strategy,primary_channel
0,Engaged Budget Shoppers,0.964,0.716125,20,"Deal-focused campaigns, bundle offers, persona...",push_notification
1,High-Value One-Time Buyers,0.644,0.543936,20,Reactivation campaigns with premium positioning,personalized_email
2,Inactive Budget Shoppers,0.804,0.692023,20,"Win-back discounts, simple value offers, clear...",email
3,Loyal High-Value Customers,0.804,0.522324,20,"VIP rewards, early access, premium recommendat...",email_and_app_homepage
4,Regular Shoppers,0.644,0.548625,20,"Cross-sell recommendations, seasonal edits, ev...",app_homepage


#### Save Outputs


In [20]:
campaign_output_path = os.path.join(PROCESSED_DIR, "campaign_recommendations.csv")
summary_output_path = os.path.join(PROCESSED_DIR, "segment_campaign_summary.csv")

campaign_recommendations.to_csv(campaign_output_path, index=False)
segment_campaign_summary.to_csv(summary_output_path, index=False)

print(campaign_output_path)
print(summary_output_path)
print(campaign_recommendations.shape)
print(segment_campaign_summary.shape)

../data/processed/hm/campaign_recommendations.csv
../data/processed/hm/segment_campaign_summary.csv
(100, 18)
(5, 6)


#### Output Check


In [21]:
campaign_recommendations.head()

,customer_segment,article_id,product_name,product_type,product_group,color_group,style,occasion,price_tier,inventory_level,base_product_score,segment_match_score,campaign_score,recommended_strategy,copy_angle,campaign_goal,campaign_message,promotion_channel
0,High-Value One-Time Buyers,706016001,Jade HW Skinny Denim TRS,Trousers,Garment Lower body,Black,streetwear,daily_wear,low,medium,0.940000,0.2,0.644000,Reactivation campaigns with premium positioning,"quality, exclusivity, worth coming back for",reactivation,Rediscover elevated style with Jade HW Skinny ...,personalized_email
1,High-Value One-Time Buyers,372860001,7p Basic Shaftless,Socks,Socks & Tights,Black,everyday,daily_wear,low,medium,0.653247,0.6,0.631948,Reactivation campaigns with premium positioning,"quality, exclusivity, worth coming back for",reactivation,Rediscover elevated style with 7p Basic Shaftl...,personalized_email
2,High-Value One-Time Buyers,610776002,Tilly (1),T-shirt,Garment Upper body,Black,elegant,work_or_outing,low,high,0.652752,0.6,0.631651,Reactivation campaigns with premium positioning,"quality, exclusivity, worth coming back for",reactivation,"Rediscover elevated style with Tilly (1), a el...",personalized_email
3,High-Value One-Time Buyers,464297007,Greta Thong Mynta Low 3p,Underwear bottom,Underwear,Black,everyday,daily_wear,low,high,0.556727,0.6,0.574036,Reactivation campaigns with premium positioning,"quality, exclusivity, worth coming back for",reactivation,Rediscover elevated style with Greta Thong Myn...,personalized_email
4,High-Value One-Time Buyers,156231001,Box 4p Tights,Underwear Tights,Socks & Tights,Black,everyday,daily_wear,low,high,0.533794,0.6,0.560276,Reactivation campaigns with premium positioning,"quality, exclusivity, worth coming back for",reactivation,"Rediscover elevated style with Box 4p Tights, ...",personalized_email


In [22]:
campaign_recommendations[
    [
        "customer_segment",
        "product_name",
        "style",
        "occasion",
        "price_tier",
        "inventory_level",
        "campaign_score",
        "promotion_channel",
        "campaign_message"
    ]
].sample(5, random_state=42)

,customer_segment,product_name,style,occasion,price_tier,inventory_level,campaign_score,promotion_channel,campaign_message
83,Loyal High-Value Customers,Curvy Jeggings HW Ankle,streetwear,daily_wear,low,high,0.567122,email_and_app_homepage,An exclusive pick for your next favorite look:...
53,Engaged Budget Shoppers,Simple as that Cheeky Tanga,everyday,vacation,low,high,0.671310,push_notification,Refresh your wardrobe with Simple as that Chee...
70,Regular Shoppers,Bird coat,elegant,work_or_outing,medium,high,0.524033,app_homepage,Meet Bird coat: a elegant Coat for work_or_out...
45,Engaged Budget Shoppers,Box 4p Tights,everyday,daily_wear,low,high,0.720276,push_notification,Refresh your wardrobe with Box 4p Tights — a e...
44,Engaged Budget Shoppers,Curvy Jeggings HW Ankle,streetwear,daily_wear,low,high,0.727122,push_notification,Refresh your wardrobe with Curvy Jeggings HW A...
